# 03 · Représenter le texte et les images

**Ce que fait ce carnet.** Un algorithme ne compare pas des mots ni des pixels : il compare des
nombres. Ce carnet applique les sept représentations demandées par la mission — cinq pour le texte,
deux pour l'image — et montre ce que chacune produit sur un même article.

**Ce qu'il établit.** Chaque produit existe désormais sous sept formes numériques, de 256 à 5 000
dimensions. Le carnet suivant dira laquelle rapproche les produits d'une même catégorie.

In [1]:
import sys
sys.path.insert(0, "..")
import numpy as np
import pandas as pd
pd.set_option("display.width", 160)

## L'article que nous suivons

Une montre, dont la fiche tient en trente-quatre mots. Nous la garderons sous la main dans tous les
carnets, jusqu'à sa prédiction finale.

In [2]:
from src.pipeline import LABEL_COL, TEXT_COL, load

df = load()
montre = df[df["product_name"].str.contains("V9 METAL STRAP", na=False)].iloc[0]

print(montre["product_name"])
print()
print(montre[TEXT_COL])

V9 METAL STRAP Analog Watch  - For Men

Specifications of V9 METAL STRAP Analog Watch  - For Men General Type Analog Style Code METAL STRAP Occasion Casual Ideal For Men Warranty NO Body Features Dial Shape Round Strap Color STEEL Dial Color BLACK


## Du texte brut aux jetons

Minuscules, ponctuation retirée, mots-outils anglais écartés. Rien de plus : ni racinisation, ni
correction orthographique. Tronquer les mots détruirait des références de modèles, qui sont parfois
les termes les plus discriminants d'une fiche.

In [3]:
from src.pretraitement import etapes_texte

etapes = etapes_texte(montre[TEXT_COL])
print(f"{len(etapes['mots'])} mots bruts → {len(etapes['jetons'])} jetons retenus")
print()
print(" · ".join(etapes["jetons"]))

34 mots bruts → 29 jetons retenus

specifications · metal · strap · analog · watch · men · general · type · analog · style · code · metal · strap · occasion · casual · ideal · men · warranty · body · features · dial · shape · round · strap · color · steel · dial · color · black


## Comptage simple, puis pondération

Le comptage est la référence la plus rudimentaire qu'on puisse construire : combien de fois chaque
mot apparaît. Le TF-IDF reprend ce comptage et le divise par la fréquence du terme dans tout le
corpus — un mot présent partout ne distingue rien.

L'effet se lit sur notre montre : `color`, en tête au comptage, disparaît des premiers rangs une
fois pondéré, parce qu'il apparaît dans presque toutes les fiches du catalogue.

In [4]:
from sklearn.feature_extraction.text import CountVectorizer

from src.text import vectoriseur

textes = df[TEXT_COL].tolist()

cv = CountVectorizer(lowercase=True, stop_words="english", max_features=5000, min_df=2).fit(textes)
tf = vectoriseur().fit(textes)

def sommet(vecteur, noms, n=6):
    v = vecteur.tocoo()
    return [(noms[c], round(float(d), 3)) for c, d in sorted(zip(v.col, v.data), key=lambda t: -t[1])[:n]]

noms_cv = np.array(cv.get_feature_names_out())
noms_tf = np.array(tf.get_feature_names_out())

print(f"comptage : {len(noms_cv)} mots au vocabulaire")
print("  ", sommet(cv.transform([montre[TEXT_COL]]), noms_cv))
print(f"TF-IDF   : {len(noms_tf)} termes au vocabulaire")
print("  ", sommet(tf.transform([montre[TEXT_COL]]), noms_tf))

comptage : 2444 mots au vocabulaire
   [('strap', 3.0), ('analog', 2.0), ('color', 2.0), ('dial', 2.0), ('men', 2.0), ('metal', 2.0)]
TF-IDF   : 5000 termes au vocabulaire
   [('metal', 0.272), ('strap', 0.246), ('strap analog', 0.215), ('color steel', 0.206), ('warranty body', 0.206), ('dial', 0.199)]


## Les sept représentations

Word2Vec apprend un vecteur par mot sur notre seul corpus, BERT produit une représentation qui
dépend du contexte, USE représente la phrase entière. Côté image, SIFT décrit des points
remarquables regroupés en « mots visuels », et VGG16 — privé de sa couche de classification — sert
d'extracteur.

Tout est mis en cache : le premier calcul prend plusieurs minutes, les suivants sont immédiats.

In [5]:
from src.representations import IMAGE, TEXTE, obtenir

entrees = {"texte": textes, "image": df["uniq_id"].tolist()}
formes = {}

for famille, registre in (("texte", TEXTE), ("image", IMAGE)):
    for nom in registre:
        X, secondes = obtenir(nom, entrees[famille])
        formes[nom] = X
        etat = f"{secondes:.0f} s" if secondes else "cache"
        print(f"  {nom:22s} {X.shape[0]} × {X.shape[1]:5d}  ({etat})")

  Comptage de mots       1050 ×  2444  (cache)
  Comptage + bigrammes   1050 ×  5000  (cache)


  TF-IDF                 1050 ×  5000  (cache)
  Word2Vec               1050 ×   300  (cache)
  BERT                   1050 ×   768  (cache)
  USE                    1050 ×   512  (cache)
  SIFT                   1050 ×   256  (cache)
  CNN (VGG16)            1050 ×   512  (cache)


## Ce que la montre devient

La même fiche, traduite sept fois. Les représentations lexicales sont creuses — la quasi-totalité
des dimensions vaut zéro — là où les représentations denses n'ont aucune valeur nulle.

In [6]:
i = int(np.where(df["uniq_id"].values == montre["uniq_id"])[0][0])

lignes = []
for nom, X in formes.items():
    v = X[i]
    lignes.append({
        "Représentation": nom,
        "Dimensions": X.shape[1],
        "Valeurs non nulles": int((v != 0).sum()),
        "Minimum": round(float(v.min()), 2),
        "Maximum": round(float(v.max()), 2),
    })

pd.DataFrame(lignes).sort_values("Dimensions", ascending=False)

,Représentation,Dimensions,Valeurs non nulles,Minimum,Maximum
1,Comptage + bigrammes,5000,42,0.00,3.00
2,TF-IDF,5000,42,0.00,0.27
0,Comptage de mots,2444,22,0.00,3.00
4,BERT,768,768,-3.80,1.19
5,USE,512,512,-0.07,0.07
7,CNN (VGG16),512,428,0.00,5.40
3,Word2Vec,300,300,-1.02,1.33
6,SIFT,256,200,0.00,0.02


**Ce que ce carnet établit.** Sept représentations disponibles, calculées sur le même corpus et
mises en cache. Aucune étiquette n'est intervenue : elles ne serviront qu'à mesurer, dans le carnet
suivant.